# Autocorrelación espacial local en Cajamarca, Amazonas y San Martín
**Evaluación final individual | Introducción a Datos Espaciales**  
José Armando Bautista Janampa · 20220430

Se analiza la distribución distrital del porcentaje de población que completó la educación secundaria en 2019. Para ello, se construyen cuatro matrices de vecindad y se aplica el índice local de Moran (LISA), que distingue agrupamientos de valores altos y bajos y valores atípicos respecto de los distritos vecinos.

El análisis comprende exclusivamente los distritos de **Cajamarca, Amazonas y San Martín**. Se utiliza la capa `good_geom` del conjunto de datos presentado en clase para evitar los problemas de superposición de la capa `distritos`.

## 1. Preparación del entorno

Se emplean GeoPandas para los datos geográficos, `libpysal` para construir las relaciones de vecindad y `esda` para calcular la autocorrelación espacial. La instalación se realiza desde el propio cuaderno.

In [ ]:
%pip install -q "geopandas>=1.0,<2" "shapely>=2,<3" "libpysal>=4.12,<5" "esda>=2.6,<3" "pyogrio>=0.9,<1" "scipy>=1.11,<2" "matplotlib>=3.8,<4" "requests>=2.31,<3"

In [ ]:
from pathlib import Path
import unicodedata
import zipfile

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyogrio
import requests
from IPython.display import display
from libpysal.graph import Graph
from matplotlib.patches import Patch
from scipy.spatial import cKDTree
import esda

SEMILLA = 42
PERMUTACIONES = 999
ALFA = 0.05
K_VECINOS = 3
CRS_TRABAJO = 32718  # WGS 84 / UTM zona 18 sur; unidades en metros
DEPARTAMENTOS = {"CAJAMARCA", "AMAZONAS", "SAN MARTIN"}
VARIABLE = "Educ_sec_comp2019_pct"

BASE = Path.cwd()
DATOS = BASE / "datos_lisa"
SALIDAS = BASE / "resultados_lisa"
DATOS.mkdir(parents=True, exist_ok=True)
SALIDAS.mkdir(parents=True, exist_ok=True)

print("Carpeta de datos:", DATOS.resolve())
print("Carpeta de resultados:", SALIDAS.resolve())

## 2. Descarga y lectura de los datos

La información se obtiene del repositorio `CienciaDeDatosEspacial/dataSets`, empleado en la última sesión. Se utiliza la capa `good_geom`, que incorpora el indicador educativo y las geometrías distritales corregidas. El cuaderno identifica el archivo GeoPackage del repositorio; si ya se descargó previamente, reutiliza el archivo local.

La variable `Educ_sec_comp2019_pct` representa el porcentaje de población que terminó la secundaria en 2019. Los resultados corresponden a esa información histórica, no a una medición actual.

In [ ]:
REPOSITORIO = "CienciaDeDatosEspacial/dataSets"
CAPA = "good_geom"
EXTENSIONES = {".gpkg", ".sqlite", ".db", ".zip"}


def capas_disponibles(ruta):
    try:
        return set(pyogrio.list_layers(ruta)[:, 0].astype(str))
    except Exception:
        return set()


def localizar_base_local():
    for ruta in sorted(DATOS.iterdir()):
        if ruta.is_file() and CAPA in capas_disponibles(ruta):
            return ruta
    return None


def descargar_base():
    local = localizar_base_local()
    if local is not None:
        return local

    sesion = requests.Session()
    sesion.headers.update({"User-Agent": "evaluacion-espacial-academica", "Accept": "application/vnd.github+json"})
    ultimo_error = None
    for rama in ("main", "master"):
        api = f"https://api.github.com/repos/{REPOSITORIO}/git/trees/{rama}?recursive=1"
        try:
            respuesta = sesion.get(api, timeout=(15, 60))
            if respuesta.status_code == 404:
                continue
            respuesta.raise_for_status()
            arbol = respuesta.json()
            if arbol.get("truncated"):
                raise RuntimeError("El inventario remoto de archivos está incompleto.")
            archivos = [f for f in arbol.get("tree", [])
                        if f.get("type") == "blob"
                        and Path(f["path"]).suffix.lower() in EXTENSIONES
                        and f.get("size", 0) <= 120_000_000]
            archivos.sort(key=lambda f: (
                not any(s in f["path"].lower() for s in ("peru", "distrit", "social")),
                Path(f["path"]).suffix.lower() == ".zip",
                f.get("size", 0)))

            for pos, archivo in enumerate(archivos, 1):
                ruta_remota = archivo["path"]
                destino = DATOS / f"{pos:02d}_{Path(ruta_remota).name}"
                if not destino.exists():
                    url = f"https://raw.githubusercontent.com/{REPOSITORIO}/{rama}/{ruta_remota}"
                    try:
                        with sesion.get(url, timeout=(15, 150), stream=True) as r:
                            r.raise_for_status()
                            temporal = destino.with_name(destino.name + ".part")
                            with temporal.open("wb") as f:
                                for fragmento in r.iter_content(chunk_size=1024 * 1024):
                                    if fragmento:
                                        f.write(fragmento)
                            temporal.replace(destino)
                    except requests.RequestException as error:
                        ultimo_error = error
                        destino.with_name(destino.name + ".part").unlink(missing_ok=True)
                        continue

                if CAPA in capas_disponibles(destino):
                    print("Archivo del repositorio:", ruta_remota)
                    return destino

                # Algunos archivos GeoPackage pueden estar comprimidos en ZIP.
                if destino.suffix.lower() == ".zip":
                    try:
                        with zipfile.ZipFile(destino) as z:
                            for miembro in z.infolist():
                                if Path(miembro.filename).suffix.lower() != ".gpkg":
                                    continue
                                extraido = DATOS / f"{pos:02d}_{Path(miembro.filename).name}"
                                if not extraido.exists():
                                    with z.open(miembro) as origen, extraido.open("wb") as salida:
                                        import shutil
                                        shutil.copyfileobj(origen, salida)
                                if CAPA in capas_disponibles(extraido):
                                    print("Archivo comprimido del repositorio:", ruta_remota)
                                    return extraido
                    except (zipfile.BadZipFile, OSError):
                        pass
        except (requests.RequestException, RuntimeError) as error:
            ultimo_error = error

    raise FileNotFoundError(
        "No se pudo localizar la capa 'good_geom' en el repositorio. "
        "Compruebe la conexión o coloque el GeoPackage utilizado en clase "
        "dentro de 'datos_lisa' y vuelva a ejecutar esta celda. "
        f"Detalle: {ultimo_error}"
    )


RUTA_BASE = descargar_base()
peru = gpd.read_file(RUTA_BASE, layer=CAPA)
print("Archivo:", RUTA_BASE.name)
print("Registros de la capa:", len(peru))
print("CRS de origen:", peru.crs)
print("Columnas:", ", ".join(peru.columns))

## 3. Delimitación del área de estudio

Se seleccionan únicamente los tres departamentos asignados. Los nombres se normalizan para reconocer variantes con o sin tilde. Las geometrías se reproyectan a UTM 18S, de modo que las distancias utilizadas en los métodos de proximidad se expresen en metros.

In [ ]:
def normalizar(texto):
    limpio = unicodedata.normalize("NFKD", str(texto))
    return "".join(c for c in limpio if not unicodedata.combining(c)).strip().upper()


COLUMNA_DEPARTAMENTO = next(
    (c for c in ("DEPARTAMENTO", "DEPARTAMEN") if c in peru.columns), None
)
if COLUMNA_DEPARTAMENTO is None:
    raise KeyError("No se encontró la columna de departamento en la capa 'good_geom'.")
if VARIABLE not in peru.columns:
    raise KeyError(f"La capa 'good_geom' no contiene la variable {VARIABLE!r}.")
if peru.crs is None:
    raise ValueError("La capa no declara un CRS; no es posible calcular distancias confiables.")

zona_original = peru.loc[
    peru[COLUMNA_DEPARTAMENTO].map(normalizar).isin(DEPARTAMENTOS)
].copy()
if zona_original.empty:
    raise ValueError("El filtro no devolvió distritos de los tres departamentos.")

zona_original["dep_estudio"] = zona_original[COLUMNA_DEPARTAMENTO].map(normalizar)
zona_original[VARIABLE] = pd.to_numeric(zona_original[VARIABLE], errors="coerce")
sin_indicador = int(zona_original[VARIABLE].isna().sum())
sin_geometria = int(zona_original.geometry.isna().sum() +
                   zona_original.geometry.is_empty.fillna(False).sum())
zona = zona_original.loc[
    zona_original[VARIABLE].notna()
    & zona_original.geometry.notna()
    & ~zona_original.geometry.is_empty
].copy()

if not zona[VARIABLE].between(0, 100).all():
    raise ValueError("Se detectaron porcentajes fuera del intervalo 0–100.")
if not zona.geometry.is_valid.all():
    raise ValueError("Existen geometrías inválidas en 'good_geom'; es necesaria una revisión espacial.")

zona = zona.to_crs(CRS_TRABAJO).reset_index(drop=True)
if len(zona) <= K_VECINOS + 1:
    raise ValueError("No existen suficientes distritos para el análisis de vecindad.")

resumen_departamentos = zona.groupby("dep_estudio").agg(
    distritos=(VARIABLE, "size"),
    educacion_media_pct=(VARIABLE, "mean")
).reindex(sorted(DEPARTAMENTOS))

print("Distritos seleccionados:", len(zona))
print("Registros sin indicador:", sin_indicador)
print("Registros sin geometría:", sin_geometria)
print("CRS de trabajo:", zona.crs)
display(resumen_departamentos.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
zona.plot(column=VARIABLE, cmap="YlOrRd", ax=ax, legend=True,
          legend_kwds={"label": "Secundaria completa (%)", "shrink": 0.65},
          linewidth=0.15, edgecolor="#777777")
ax.set_title("Educación secundaria completa (2019)\nCajamarca, Amazonas y San Martín", fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(SALIDAS / "00_Indicador_educacion.png", dpi=220, bbox_inches="tight")
plt.show()

## 4. Construcción de cuatro criterios de vecindad

La identificación de vecinos se realiza con cuatro reglas:

1. **Rook:** polígonos que comparten un tramo de frontera.
2. **Queen:** polígonos que comparten un tramo de frontera o al menos un vértice.
3. **KNN (k = 3):** los tres puntos representativos más próximos a cada distrito.
4. **Banda de distancia:** puntos representativos separados por una distancia menor o igual que un umbral común.

En la banda de distancia, el umbral corresponde a la mayor distancia al vecino más próximo dentro de la zona de estudio. Así se evita que la banda deje distritos aislados. Se usan puntos representativos, que siempre pertenecen a sus polígonos.

La contigüidad Rook puede producir distritos sin ningún vecino, especialmente cuando existen separaciones o discontinuidades geométricas. Para que las cuatro comparaciones utilicen exactamente las mismas unidades espaciales, esos distritos se apartan de los cuatro cálculos y se registra su cantidad. Las matrices restantes se normalizan por filas: los pesos de los vecinos de cada distrito suman 1.

In [ ]:
rook_inicial = Graph.build_contiguity(zona, rook=True)
indices_aislados = list(rook_inicial.isolates)
excluidos = zona.loc[indices_aislados].copy()
analisis = zona.drop(index=indices_aislados).reset_index(drop=True)

if len(analisis) <= K_VECINOS + 1:
    raise ValueError("Se perdieron demasiadas observaciones al excluir aislados Rook.")

puntos = analisis.representative_point()
xy = np.column_stack([puntos.x.to_numpy(), puntos.y.to_numpy()])
distancias_nn, _ = cKDTree(xy).query(xy, k=2)
UMBRAL_BANDA = float(distancias_nn[:, 1].max() + 1.0)  # Margen de 1 metro.

GRAFOS = {
    "Rook": Graph.build_contiguity(analisis, rook=True).transform("r"),
    "Queen": Graph.build_contiguity(analisis, rook=False).transform("r"),
    "KNN_3": Graph.build_knn(puntos, k=K_VECINOS).transform("r"),
    "Banda_distancia": Graph.build_distance_band(
        puntos, threshold=UMBRAL_BANDA, binary=True
    ).transform("r"),
}

for nombre, grafo in GRAFOS.items():
    if len(grafo.isolates) != 0:
        raise RuntimeError(f"El método {nombre} presenta distritos aislados: {list(grafo.isolates)}")
    if grafo.n != len(analisis):
        raise RuntimeError(f"El método {nombre} no utiliza la misma cantidad de distritos.")

print("Distritos del área de estudio:", len(zona))
print("Distritos aislados por Rook (excluidos en los cuatro análisis):", len(excluidos))
print("Distritos comparados:", len(analisis))
print(f"Banda de distancia: {UMBRAL_BANDA/1000:.2f} km")

vecinos = pd.DataFrame({
    nombre: g.cardinalities.to_numpy() for nombre, g in GRAFOS.items()
})
display(vecinos.agg(["min", "mean", "median", "max"]).round(2))

if len(excluidos):
    columnas = [c for c in ("dep_estudio", "PROVINCIA", "DISTRITO", "ubigeo", "Ubigeo")
                if c in excluidos.columns]
    display(excluidos[columnas].reset_index(drop=True))

## 5. Autocorrelación espacial global

El índice I de Moran evalúa la asociación entre el indicador de cada distrito y los valores de sus vecinos. Los valores positivos indican similitud espacial y los negativos, contraste espacial. Se calcula su significancia mediante 999 permutaciones aleatorias. Un valor `p_sim` menor que 0.05 indica evidencia estadística frente a la hipótesis de ausencia de autocorrelación espacial.

También se calcula el rezago espacial, que corresponde al promedio ponderado del indicador en los distritos vecinos.

In [ ]:
Y = analisis[VARIABLE].to_numpy(dtype=float)
RESULTADOS = {}
filas_globales = []

for nombre, grafo in GRAFOS.items():
    np.random.seed(SEMILLA)
    global_moran = esda.Moran(Y, grafo, permutations=PERMUTACIONES)
    local_moran = esda.Moran_Local(
        Y, grafo, permutations=PERMUTACIONES, seed=SEMILLA, n_jobs=1
    )
    rezago = np.asarray(grafo.lag(analisis[VARIABLE]), dtype=float)
    etiquetas = local_moran.get_cluster_labels(crit_value=ALFA)

    resultado = analisis.copy()
    resultado["rezago_espacial"] = rezago
    resultado["I_local"] = np.asarray(local_moran.Is, dtype=float)
    resultado["p_local"] = np.asarray(local_moran.p_sim, dtype=float)
    resultado["LISA"] = etiquetas

    RESULTADOS[nombre] = {
        "datos": resultado,
        "global": global_moran,
        "local": local_moran,
    }
    filas_globales.append({
        "Metodo": nombre,
        "Distritos": len(resultado),
        "Vecinos_promedio": float(grafo.cardinalities.mean()),
        "Moran_I": float(global_moran.I),
        "p_sim": float(global_moran.p_sim),
    })

TABLA_GLOBAL = pd.DataFrame(filas_globales).set_index("Metodo")
display(TABLA_GLOBAL.round(4))

## 6. Mapas de autocorrelación espacial local (LISA)

Para cada matriz de vecindad se clasifican los distritos según el índice local de Moran, con un nivel de significancia del 5 %. La clasificación distingue:

- **hotSpot (High-High):** valor alto rodeado de valores altos.
- **coldSpot (Low-Low):** valor bajo rodeado de valores bajos.
- **hotOutlier (High-Low):** valor alto rodeado de valores bajos.
- **coldOutlier (Low-High):** valor bajo rodeado de valores altos.
- **no_pattern:** asociación local no significativa.

Se conserva la misma simbología y la misma extensión geográfica en los cuatro mapas para facilitar la comparación. Los distritos excluidos por falta de vecinos Rook se muestran diferenciados, sin asignarles una categoría LISA.

In [ ]:
ETIQUETAS = {
    "High-Low": "1 hotOutlier",
    "High-High": "2 hotSpot",
    "Insignificant": "3 no_pattern",
    "Low-Low": "4 coldSpot",
    "Low-High": "5 coldOutlier",
}
# Paleta divergente PuOr del ejercicio de clase, con una clase neutra central.
PALETA = plt.get_cmap("PuOr", 5)
COLORES = {etiqueta: PALETA(i) for i, etiqueta in enumerate(ETIQUETAS.values())}
ORDEN = list(ETIQUETAS.values())

for nombre, contenido in RESULTADOS.items():
    datos = contenido["datos"]
    datos["LISA_clase"] = datos["LISA"].map(ETIQUETAS)
    if datos["LISA_clase"].isna().any():
        raise ValueError(f"Se encontraron categorías LISA desconocidas para {nombre}.")
    datos["LISA_clase"] = pd.Categorical(datos["LISA_clase"], categories=ORDEN,
                                          ordered=True)


def dibujar_mapa(nombre, contenido):
    datos = contenido["datos"]
    global_moran = contenido["global"]
    fig, ax = plt.subplots(figsize=(9, 10))

    # Se conserva la extensión original de los tres departamentos.
    zona.plot(ax=ax, facecolor="#ffffff", edgecolor="#dddddd", linewidth=0.20)
    datos.plot(ax=ax, color=[COLORES[str(clase)] for clase in datos["LISA_clase"]],
               edgecolor="#8a8a8a", linewidth=0.14)
    if len(excluidos):
        excluidos.plot(ax=ax, facecolor="#d5d5d5", edgecolor="#4a4a4a",
                       linewidth=0.35, hatch="///")

    margen = 14000
    xmin, ymin, xmax, ymax = zona.total_bounds
    ax.set_xlim(xmin - margen, xmax + margen)
    ax.set_ylim(ymin - margen, ymax + margen)
    ax.set_aspect("equal")
    ax.set_axis_off()
    titulo = nombre.replace("_", " ")
    ax.set_title(f"Mapa LISA: {titulo}\nCajamarca, Amazonas y San Martín",
                 fontsize=14, weight="bold", pad=18)

    elementos = [Patch(facecolor=COLORES[c], edgecolor="#888888", label=c) for c in ORDEN]
    if len(excluidos):
        elementos.append(Patch(facecolor="#d5d5d5", edgecolor="#4a4a4a",
                               hatch="///", label="Sin vecinos Rook"))
    ax.legend(handles=elementos, loc="lower left", fontsize=9,
              frameon=True, title="Clasificación LISA")
    fig.text(0.5, 0.05,
             f"Educación secundaria completa, 2019 | α = {ALFA:.2f} | "
             f"I de Moran = {global_moran.I:.3f} | p = {global_moran.p_sim:.3f}",
             ha="center", fontsize=9)
    fig.text(0.5, 0.028, "Fuente: CienciaDeDatosEspacial/dataSets; elaboración propia.",
             ha="center", fontsize=8, color="#555555")
    fig.subplots_adjust(bottom=0.09, top=0.91)
    ruta = SALIDAS / f"Mapa_LISA_{nombre}.png"
    fig.savefig(ruta, dpi=240, bbox_inches="tight", facecolor="white")
    plt.show()
    return ruta


ARCHIVOS_MAPA = {}
for metodo, contenido in RESULTADOS.items():
    ARCHIVOS_MAPA[metodo] = dibujar_mapa(metodo, contenido)

print("Mapas exportados:")
for ruta in ARCHIVOS_MAPA.values():
    print(" -", ruta.name)

## 7. Comparación de los cuatro métodos

La comparación considera el número de distritos clasificados en cada categoría y la autocorrelación global obtenida. Las diferencias entre los mapas corresponden a cambios en la definición de vecindad, ya que se mantiene el mismo indicador y el mismo conjunto de distritos.

In [ ]:
conteos = pd.DataFrame({
    metodo: contenido["datos"]["LISA_clase"].value_counts()
    for metodo, contenido in RESULTADOS.items()
}).reindex(ORDEN).fillna(0).astype(int)
conteos.loc["Total"] = conteos.sum(axis=0)

print("Distritos por categoría LISA:")
display(conteos)
print("Autocorrelación global por método:")
display(TABLA_GLOBAL.round(4))

## 8. Exportación y verificación

Se guardan los cuatro mapas en PNG, los resultados distritales en GeoPackage para su consulta en QGIS y las tablas comparativas en CSV. Finalmente, se comprueba que cada mapa tenga una sola clasificación por distrito, que las categorías sean válidas y que las matrices de pesos estén normalizadas.

In [ ]:
TABLA_GLOBAL.to_csv(SALIDAS / "Moran_global.csv", encoding="utf-8-sig")
conteos.to_csv(SALIDAS / "Conteos_LISA.csv", encoding="utf-8-sig")

for nombre, grafo in GRAFOS.items():
    datos = RESULTADOS[nombre]["datos"].copy()
    datos["LISA_clase"] = datos["LISA_clase"].astype(str)
    datos.to_file(SALIDAS / f"LISA_{nombre}.gpkg", layer="resultados", driver="GPKG")

    assert len(datos) == len(analisis)
    assert datos["LISA_clase"].isin(ORDEN).all()
    assert int(conteos.loc["Total", nombre]) == len(analisis)
    assert np.isfinite(datos["p_local"]).all()
    assert np.all((datos["p_local"] >= 0) & (datos["p_local"] <= 1))
    assert np.allclose(grafo.adjacency.groupby(level="focal").sum().to_numpy(), 1.0,
                       atol=1e-10)
    assert ARCHIVOS_MAPA[nombre].is_file()

assert len(GRAFOS) == 4 and len(ARCHIVOS_MAPA) == 4
assert len(analisis) + len(excluidos) == len(zona)

print("Verificación satisfactoria: cuatro grafos, cuatro mapas y clasificaciones completas.")
print("Resultados guardados en:", SALIDAS.resolve())

## 9. Alcance de los resultados

Los mapas identifican asociaciones espaciales de un indicador educativo correspondiente a 2019. Los agrupamientos significativos no demuestran causalidad ni permiten atribuir sus patrones a un factor específico. Su delimitación depende del criterio de vecindad y del nivel de significancia adoptado.

La comparación se limita a Cajamarca, Amazonas y San Martín. Por tanto, los distritos situados en el límite del área de estudio no consideran como vecinos a los distritos de departamentos no seleccionados. La cantidad de unidades descartadas por aislamiento Rook se registra en la sección 4.

**Datos y referencia metodológica:** [CienciaDeDatosEspacial/dataSets](https://github.com/CienciaDeDatosEspacial/dataSets) y material de clase [Neighborhood and Spatial Correlation](https://cienciadedatosespacial.github.io/spatial_autoCorr/).